# Chuẩn bị dữ liệu

## Đọc NHANES

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, "..")

from importlib import reload
import src.utils as utils

reload(utils)
from src.utils import load_xpt, get_smoking_status, get_physical_activity, get_yes_no, get_diabetes_label

data_dir = Path("../data/raw/nhanes")
from src.utils import map_nhanes_inputs, map_brfss_inputs, get_brfss_diabetes_label


In [2]:
demo_i = load_xpt(data_dir / "2015-2016/DEMO_I.xpt")
demo_j = load_xpt(data_dir / "2017-2018/DEMO_J.xpt")

demo_i.shape, demo_j.shape


((9971, 47), (9254, 46))

In [3]:
cols = ["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH3"]

demo_i[cols].head()


,SEQN,RIDAGEYR,RIAGENDR,RIDRETH3
0,83732.0,62.0,1.0,3.0
1,83733.0,53.0,1.0,3.0
2,83734.0,78.0,1.0,3.0
3,83735.0,56.0,2.0,3.0
4,83736.0,42.0,2.0,4.0


In [4]:
demo_i["SEQN"].duplicated().sum(), demo_j["SEQN"].duplicated().sum()


(np.int64(0), np.int64(0))

In [5]:
set(demo_i.columns) - set(demo_j.columns), \
set(demo_j.columns) - set(demo_i.columns)


({'DMDHRAGE', 'DMDHRBR4', 'DMDHREDU', 'DMDHRMAR', 'DMDHSEDU'},
 {'DMDHRAGZ', 'DMDHREDZ', 'DMDHRMAZ', 'DMDHSEDZ'})

In [6]:
cols = [
    "SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH3",
    "WTINT2YR", "WTMEC2YR", "SDMVPSU", "SDMVSTRA"
]

demo_i = demo_i[cols].copy()
demo_j = demo_j[cols].copy()


## Nối chẩn đoán

DIQ070 theo skip của DIQ010/DIQ160; thiếu do không được hỏi vẫn là thiếu.

In [7]:
diq_i = pd.read_sas(
    data_dir / "2015-2016/DIQ_I.xpt", format="xport"
)
diq_j = pd.read_sas(
    data_dir / "2017-2018/DIQ_J.xpt", format="xport"
)

diq_i.shape, diq_j.shape


((9575, 54), (8897, 54))

In [8]:
cols = ["SEQN", "DIQ010", "DIQ050", "DIQ070"]

diq_i[cols].head()


,SEQN,DIQ010,DIQ050,DIQ070
0,83732.0,1.0,1.0,1.0
1,83733.0,2.0,2.0,NaN
2,83734.0,1.0,2.0,2.0
3,83735.0,2.0,2.0,2.0
4,83736.0,2.0,2.0,NaN


In [9]:
diq_i["SEQN"].duplicated().sum(), diq_j["SEQN"].duplicated().sum()


(np.int64(0), np.int64(0))

In [10]:
diq_cols = ["SEQN", "DIQ010", "DIQ050", "DIQ070"]

nhanes_i = demo_i.merge(
    diq_i[diq_cols], on="SEQN", how="left", validate="one_to_one"
)

nhanes_j = demo_j.merge(
    diq_j[diq_cols], on="SEQN", how="left", validate="one_to_one"
)

nhanes_i.shape, nhanes_j.shape


((9971, 11), (9254, 11))

In [11]:
pd.DataFrame({
    "2015-2016": nhanes_i[diq_cols[1:]].isna().sum(),
    "2017-2018": nhanes_j[diq_cols[1:]].isna().sum()
})


,2015-2016,2017-2018
DIQ010,396,357
DIQ050,396,357
DIQ070,8456,7580


In [12]:
adults_i = nhanes_i.loc[nhanes_i["RIDAGEYR"] >= 18].copy()
adults_j = nhanes_j.loc[nhanes_j["RIDAGEYR"] >= 18].copy()

adults_i.shape, adults_j.shape


((5992, 11), (5856, 11))

In [13]:
cols = ["DIQ010", "DIQ050", "DIQ070"]

pd.DataFrame({
    "2015-2016": adults_i[cols].isna().sum(),
    "2017-2018": adults_j[cols].isna().sum()
})


,2015-2016,2017-2018
DIQ010,0,0
DIQ050,0,0
DIQ070,4524,4227


In [14]:
pd.crosstab(
    adults_i["DIQ010"],
    adults_i["DIQ070"].fillna("Missing")
)


DIQ070,1.0,2.0,7.0,9.0,Missing
DIQ010,,,,,
1.0,577,266,1,0,1
2.0,31,459,0,1,4520
3.0,32,100,0,0,0
9.0,0,1,0,0,3


In [15]:
pd.crosstab(
    adults_j["DIQ010"],
    adults_j["DIQ070"].fillna("Missing")
)


DIQ070,1.0,2.0,9.0,Missing
DIQ010,,,,
1.0,644,234,0,0
2.0,38,537,1,4224
3.0,33,140,1,0
9.0,0,1,0,3


In [16]:
adults_i = adults_i.merge(
    diq_i[["SEQN", "DIQ160"]],
    on="SEQN", how="left", validate="one_to_one"
)

adults_j = adults_j.merge(
    diq_j[["SEQN", "DIQ160"]],
    on="SEQN", how="left", validate="one_to_one"
)


In [17]:
for df in (adults_i, adults_j):
    eligible = df["DIQ010"].isin([1, 3]) | df["DIQ160"].eq(1)

    display(
        df.loc[
            eligible & df["DIQ070"].isna(),
            ["SEQN", "DIQ010", "DIQ160", "DIQ070"]
        ]
    )


,SEQN,DIQ010,DIQ160,DIQ070
4635,91416.0,1.0,NaN,NaN


,SEQN,DIQ010,DIQ160,DIQ070


In [18]:
for df in (adults_i, adults_j):
    eligible = df["DIQ010"].isin([1, 3]) | df["DIQ160"].eq(1)

    display(pd.Series({
        "skip_missing": (~eligible & df["DIQ070"].isna()).sum(),
        "eligible_missing": (eligible & df["DIQ070"].isna()).sum()
    }))


skip_missing        4523
eligible_missing       1
dtype: int64

skip_missing        4227
eligible_missing       0
dtype: int64

## Nối HbA1c

In [19]:
ghb_i = load_xpt(data_dir / "2015-2016/GHB_I.xpt")
ghb_j = load_xpt(data_dir / "2017-2018/GHB_J.xpt")

ghb_i.shape, ghb_j.shape


((6744, 2), (6401, 2))

In [20]:
ghb_i.columns.tolist(), ghb_j.columns.tolist()


(['SEQN', 'LBXGH'], ['SEQN', 'LBXGH'])

In [21]:
ghb_i["SEQN"].duplicated().sum(), ghb_j["SEQN"].duplicated().sum()


(np.int64(0), np.int64(0))

In [22]:
adults_i = adults_i.merge(
    ghb_i[["SEQN", "LBXGH"]],
    on="SEQN", how="left", validate="one_to_one"
)

adults_j = adults_j.merge(
    ghb_j[["SEQN", "LBXGH"]],
    on="SEQN", how="left", validate="one_to_one"
)


In [23]:
pd.DataFrame({
    "2015-2016": adults_i["LBXGH"].describe(),
    "2017-2018": adults_j["LBXGH"].describe()
})


,2015-2016,2017-2018
count,5441.000000,5261.000000
mean,5.822000,5.837179
std,1.165878,1.086560
min,3.800000,3.800000
25%,5.300000,5.300000
50%,5.500000,5.600000
75%,5.900000,6.000000
max,17.000000,16.200000


In [24]:
adults_i.shape, adults_j.shape, (
    adults_i["LBXGH"].isna().sum(),
    adults_j["LBXGH"].isna().sum()
)


((5992, 13), (5856, 13), (np.int64(551), np.int64(595)))

## Nối glucose

In [25]:
glu_i = load_xpt(data_dir / "2015-2016/GLU_I.xpt")
glu_j = load_xpt(data_dir / "2017-2018/GLU_J.xpt")


In [26]:
glu_i.columns.tolist(), glu_j.columns.tolist()


(['SEQN', 'WTSAF2YR', 'LBXGLU', 'LBDGLUSI'],
 ['SEQN', 'WTSAF2YR', 'LBXGLU', 'LBDGLUSI'])

In [27]:
pd.DataFrame({
    cycle: {
        "rows": len(df),
        "columns": len(df.columns),
        "SEQN_missing": df["SEQN"].isna().sum(),
        "SEQN_duplicates": df["SEQN"].duplicated().sum(),
        "LBXGLU_missing": df["LBXGLU"].isna().sum(),
    }
    for cycle, df in [("2015-2016", glu_i), ("2017-2018", glu_j)]
})


,2015-2016,2017-2018
rows,3191,3036
columns,4,4
SEQN_missing,0,0
SEQN_duplicates,0,0
LBXGLU_missing,219,145


In [28]:
glu_cols = ["SEQN", "LBXGLU", "WTSAF2YR"]

adults_i = adults_i.merge(
    glu_i[glu_cols],
    on="SEQN", how="left", validate="one_to_one"
)

adults_j = adults_j.merge(
    glu_j[glu_cols],
    on="SEQN", how="left", validate="one_to_one"
)


In [29]:
pd.DataFrame({
    cycle: {
        "adults": len(df),
        "LBXGLU_available": df["LBXGLU"].notna().sum(),
        "LBXGLU_missing": df["LBXGLU"].isna().sum(),
    }
    for cycle, df in [("2015-2016", adults_i), ("2017-2018", adults_j)]
})


,2015-2016,2017-2018
adults,5992,5856
LBXGLU_available,2573,2537
LBXGLU_missing,3419,3319


In [30]:
pd.DataFrame({
    cycle: {
        "glucose_available": df["LBXGLU"].notna().sum(),
        "glucose_with_positive_weight": (
            df["LBXGLU"].notna() & df["WTSAF2YR"].gt(0)
        ).sum(),
        "glucose_with_zero_weight": (
            df["LBXGLU"].notna() & df["WTSAF2YR"].eq(0)
        ).sum(),
        "glucose_with_missing_weight": (
            df["LBXGLU"].notna() & df["WTSAF2YR"].isna()
        ).sum(),
    }
    for cycle, df in [("2015-2016", adults_i), ("2017-2018", adults_j)]
})


,2015-2016,2017-2018
glucose_available,2573,2537
glucose_with_positive_weight,2379,2387
glucose_with_zero_weight,194,150
glucose_with_missing_weight,0,0


## Nối nhịn ăn

In [31]:
fast_i = load_xpt(data_dir / "2015-2016/FASTQX_I.xpt")
fast_j = load_xpt(data_dir / "2017-2018/FASTQX_J.xpt")


In [32]:
fast_i.columns.tolist(), fast_j.columns.tolist()


(['SEQN',
  'PHQ020',
  'PHACOFHR',
  'PHACOFMN',
  'PHQ030',
  'PHAALCHR',
  'PHAALCMN',
  'PHQ040',
  'PHAGUMHR',
  'PHAGUMMN',
  'PHQ050',
  'PHAANTHR',
  'PHAANTMN',
  'PHQ060',
  'PHASUPHR',
  'PHASUPMN',
  'PHAFSTHR',
  'PHAFSTMN',
  'PHDSESN'],
 ['SEQN',
  'PHQ020',
  'PHACOFHR',
  'PHACOFMN',
  'PHQ030',
  'PHAALCHR',
  'PHAALCMN',
  'PHQ040',
  'PHAGUMHR',
  'PHAGUMMN',
  'PHQ050',
  'PHAANTHR',
  'PHAANTMN',
  'PHQ060',
  'PHASUPHR',
  'PHASUPMN',
  'PHAFSTHR',
  'PHAFSTMN',
  'PHDSESN'])

In [33]:
pd.DataFrame({
    cycle: {
        "rows": len(df),
        "columns": len(df.columns),
        "SEQN_missing": df["SEQN"].isna().sum(),
        "SEQN_duplicates": df["SEQN"].duplicated().sum(),
        "PHAFSTHR_missing": df["PHAFSTHR"].isna().sum(),
        "PHAFSTMN_missing": df["PHAFSTMN"].isna().sum(),
        "PHDSESN_missing": df["PHDSESN"].isna().sum(),
    }
    for cycle, df in [("2015-2016", fast_i), ("2017-2018", fast_j)]
})


,2015-2016,2017-2018
rows,9165,8366
columns,19,19
SEQN_missing,0,0
SEQN_duplicates,0,0
PHAFSTHR_missing,254,370
PHAFSTMN_missing,254,370
PHDSESN_missing,0,0


In [34]:
glucose_fasting = {}

for cycle, adults, fast in [
    ("2015-2016", adults_i, fast_i),
    ("2017-2018", adults_j, fast_j),
]:
    check = adults.loc[
        adults["LBXGLU"].notna(), ["SEQN", "LBXGLU", "WTSAF2YR"]
    ].merge(
        fast, on="SEQN", how="left", validate="one_to_one",
        indicator="FASTQX_match"
    )
    check["food_fast_hours"] = check["PHAFSTHR"] + check["PHAFSTMN"] / 60
    glucose_fasting[cycle] = check


In [35]:
pd.DataFrame({
    cycle: {
        "with_glucose": len(df),
        "FASTQX_unmatched": df["FASTQX_match"].eq("left_only").sum(),
        "food_fast_missing": df["food_fast_hours"].isna().sum(),
        "morning_session": df["PHDSESN"].eq(0).sum(),
        "afternoon_or_evening_session": df["PHDSESN"].isin([1, 2]).sum(),
        "session_missing": df["PHDSESN"].isna().sum(),
        "food_fast_lt8": df["food_fast_hours"].lt(8).sum(),
        "food_fast_8_to_lt9": df["food_fast_hours"].between(8, 9, inclusive="left").sum(),
        "food_fast_9_to_lt24": df["food_fast_hours"].between(9, 24, inclusive="left").sum(),
        "food_fast_ge24": df["food_fast_hours"].ge(24).sum(),
    }
    for cycle, df in glucose_fasting.items()
})


,2015-2016,2017-2018
with_glucose,2573,2537
FASTQX_unmatched,0,0
food_fast_missing,0,0
morning_session,2573,2537
afternoon_or_evening_session,0,0
session_missing,0,0
food_fast_lt8,180,123
food_fast_8_to_lt9,6,2
food_fast_9_to_lt24,2373,2385
food_fast_ge24,14,27


## Glucose dùng được

Cờ True cần đủ các điều kiện dưới đây; thiếu thông tin bắt buộc nhận NA, kể cả khi điều kiện khác không đạt. False không có nghĩa glucose thấp.

| Thành phần | Điều kiện |
|---|---|
| Xét nghiệm | Có LBXGLU, khám buổi sáng |
| Nhịn ăn | 8–<24 giờ |
| Trọng số | WTSAF2YR > 0 |
| Đồ uống | Không dùng cà phê/trà có sữa hoặc đường hay đồ uống có cồn trong <8 giờ |

Câu đồ uống trả lời Không bỏ qua giờ/phút theo skip; trả lời Có cần đủ giờ/phút. Các cờ thuốc, kẹo và thực phẩm bổ sung trong FASTQX chỉ dùng cho phân tích độ nhạy.

In [ ]:
from src.utils import get_glucose_usable

glucose_checked = {}
for cycle, adults, fast in [
    ("2015-2016", adults_i, fast_i),
    ("2017-2018", adults_j, fast_j),
]:
    checked = adults.merge(fast, on="SEQN", how="left", validate="one_to_one")
    checked["glucose_usable"] = get_glucose_usable(checked)
    glucose_checked[cycle] = checked


In [ ]:
glucose_summary = []
for cycle, df in glucose_checked.items():
    for status, mask in [
        ("usable", df["glucose_usable"].eq(True)),
        ("unusable", df["glucose_usable"].eq(False)),
        ("unknown", df["glucose_usable"].isna()),
    ]:
        glucose_summary.append({
            "cycle": cycle,
            "status": status,
            "people": int(mask.sum()),
            "glucose_ge126": int((mask & df["LBXGLU"].ge(126)).sum()),
            "glucose_missing": int((mask & df["LBXGLU"].isna()).sum()),
        })
pd.DataFrame(glucose_summary)


## Nối BMI

In [ ]:
bmx_i = load_xpt(data_dir / "2015-2016/BMX_I.xpt", columns=["SEQN", "BMXBMI"])
bmx_j = load_xpt(data_dir / "2017-2018/BMX_J.xpt", columns=["SEQN", "BMXBMI"])

for bmx in (bmx_i, bmx_j):
    assert bmx["SEQN"].notna().all(), "Missing SEQN in BMX"
    assert bmx["SEQN"].is_unique, "Duplicate SEQN in BMX"

pd.DataFrame({
    cycle: {
        "rows": len(bmx),
        "SEQN_missing": int(bmx["SEQN"].isna().sum()),
        "SEQN_duplicates": int(bmx["SEQN"].duplicated().sum()),
    }
    for cycle, bmx in [("2015-2016", bmx_i), ("2017-2018", bmx_j)]
})


                 2015-2016  2017-2018
rows                  9544       8704
SEQN_missing             0          0
SEQN_duplicates          0          0


In [ ]:
bmi_adults = {}
bmi_summary = []
for cycle, adults, bmx in [
    ("2015-2016", adults_i, bmx_i),
    ("2017-2018", adults_j, bmx_j),
]:
    before = len(adults)
    already_present = "BMXBMI" in adults.columns
    merged = adults if already_present else adults.merge(
        bmx, on="SEQN", how="left", validate="one_to_one"
    )
    assert len(merged) == before
    bmi_adults[cycle] = merged
    bmi_summary.append({
        "cycle": cycle,
        "BMI_already_present": already_present,
        "rows_before": before,
        "rows_after": len(merged),
        "BMI_missing": int(merged["BMXBMI"].isna().sum()),
    })

adults_i = bmi_adults["2015-2016"]
adults_j = bmi_adults["2017-2018"]
pd.DataFrame(bmi_summary).set_index("cycle")


           BMI_already_present  rows_before  rows_after  BMI_missing
cycle                                                               
2015-2016                False         5992        5992          330
2017-2018                False         5856        5856          422


In [ ]:
pd.DataFrame({
    "2015-2016": adults_i["BMXBMI"].describe(),
    "2017-2018": adults_j["BMXBMI"].describe(),
})


         2015-2016    2017-2018
count  5662.000000  5434.000000
mean     29.382197    29.691700
std       7.095921     7.440481
min      14.500000    14.200000
25%      24.300000    24.600000
50%      28.300000    28.500000
75%      33.000000    33.500000
max      67.300000    86.200000


## Nối hút thuốc

In [ ]:
smq_cols = ["SEQN", "SMQ020", "SMQ040"]
smq_i = load_xpt(data_dir / "2015-2016/SMQ_I.xpt", columns=smq_cols)
smq_j = load_xpt(data_dir / "2017-2018/SMQ_J.xpt", columns=smq_cols)

for smq in (smq_i, smq_j):
    assert smq["SEQN"].notna().all(), "Missing SEQN in SMQ"
    assert smq["SEQN"].is_unique, "Duplicate SEQN in SMQ"
    assert smq["SMQ020"].dropna().isin([1, 2, 7, 9]).all(), "Unexpected SMQ020 code"
    assert smq["SMQ040"].dropna().isin([1, 2, 3, 7, 9]).all(), "Unexpected SMQ040 code"

pd.DataFrame({
    cycle: {
        "rows": len(smq),
        "SEQN_missing": int(smq["SEQN"].isna().sum()),
        "SEQN_duplicates": int(smq["SEQN"].duplicated().sum()),
    }
    for cycle, smq in [("2015-2016", smq_i), ("2017-2018", smq_j)]
})


                 2015-2016  2017-2018
rows                  7001       6724
SEQN_missing             0          0
SEQN_duplicates          0          0

In [ ]:
smoking_adults = {}
smoking_summary = []
for cycle, adults, smq in [
    ("2015-2016", adults_i, smq_i),
    ("2017-2018", adults_j, smq_j),
]:
    before = len(adults)
    missing_cols = [c for c in ["SMQ020", "SMQ040"] if c not in adults.columns]
    merged = adults if not missing_cols else adults.merge(
        smq[["SEQN"] + missing_cols],
        on="SEQN", how="left", validate="one_to_one"
    )
    assert len(merged) == before
    smoking_adults[cycle] = merged
    smoking_summary.append({
        "cycle": cycle,
        "SMQ_already_present": not missing_cols,
        "rows_before": before,
        "rows_after": len(merged),
    })

adults_i = smoking_adults["2015-2016"]
adults_j = smoking_adults["2017-2018"]
pd.DataFrame(smoking_summary).set_index("cycle")


           SMQ_already_present  rows_before  rows_after
cycle                                                  
2015-2016                False         5992        5992
2017-2018                False         5856        5856

In [ ]:
smoking_crosstabs = {
    cycle: pd.crosstab(
        adults["SMQ020"].fillna("Missing"),
        adults["SMQ040"].fillna("Missing"),
    ).reindex(
        index=[1, 2, 7, 9, "Missing"],
        columns=[1, 2, 3, 7, 9, "Missing"],
        fill_value=0,
    )
    for cycle, adults in [("2015-2016", adults_i), ("2017-2018", adults_j)]
}
pd.concat(smoking_crosstabs, names=["cycle"])


SMQ040               1    2     3  7  9  Missing
cycle     SMQ020                                
2015-2016 1        832  268  1322  0  0        0
          2          0    0     0  0  0     3559
          7          0    0     0  0  0        2
          9          0    0     0  0  0        9
          Missing    0    0     0  0  0        0
2017-2018 1        805  216  1338  0  0        0
          2          0    0     0  0  0     3497
          7          0    0     0  0  0        0
          9          0    0     0  0  0        0
          Missing    0    0     0  0  0        0

In [ ]:
smoking_status_checks = []
for cycle, adults in [("2015-2016", adults_i), ("2017-2018", adults_j)]:
    before = len(adults)
    smq_before = adults[["SMQ020", "SMQ040"]].copy()
    adults["smoking_status"] = get_smoking_status(adults)
    assert len(adults) == before
    pd.testing.assert_frame_equal(adults[["SMQ020", "SMQ040"]], smq_before)
    smoking_status_checks.append({
        "cycle": cycle, "rows_before": before, "rows_after": len(adults)
    })

pd.DataFrame(smoking_status_checks).set_index("cycle")


           rows_before  rows_after
cycle                             
2015-2016         5992        5992
2017-2018         5856        5856

In [ ]:
pd.DataFrame({
    "2015-2016": adults_i["smoking_status"].value_counts(dropna=False),
    "2017-2018": adults_j["smoking_status"].value_counts(dropna=False),
}).reindex(["never", "former", "current", pd.NA], fill_value=0).fillna(0).astype(int)


                2015-2016  2017-2018
smoking_status                      
never                3559       3497
former               1322       1338
current              1100       1021
<NA>                   11          0

## Nối tiền sử bệnh

CDC MCQ_I/J chỉ hỏi từ 20 tuổi; thiếu ở người 18–19 tuổi không phải câu trả lời Không.

In [ ]:
mcq_cols = ["SEQN", "MCQ160E", "MCQ160F"]
mcq_i = load_xpt(data_dir / "2015-2016/MCQ_I.xpt", columns=mcq_cols)
mcq_j = load_xpt(data_dir / "2017-2018/MCQ_J.xpt", columns=mcq_cols)

for mcq in (mcq_i, mcq_j):
    assert mcq["SEQN"].notna().all(), "Missing SEQN in MCQ"
    assert mcq["SEQN"].is_unique, "Duplicate SEQN in MCQ"
    assert mcq[["MCQ160E", "MCQ160F"]].isin([1, 2, 7, 9]).where(
        mcq[["MCQ160E", "MCQ160F"]].notna(), True
    ).all().all(), "Unexpected MCQ response code"

pd.DataFrame({
    cycle: {
        "rows": len(mcq),
        "SEQN_missing": int(mcq["SEQN"].isna().sum()),
        "SEQN_duplicates": int(mcq["SEQN"].duplicated().sum()),
    }
    for cycle, mcq in [("2015-2016", mcq_i), ("2017-2018", mcq_j)]
})


                 2015-2016  2017-2018
rows                  9575       8897
SEQN_missing             0          0
SEQN_duplicates          0          0

In [ ]:
mcq_adults = {}
mcq_summary = []
for cycle, adults, mcq in [
    ("2015-2016", adults_i, mcq_i),
    ("2017-2018", adults_j, mcq_j),
]:
    before = len(adults)
    missing_cols = [c for c in ["MCQ160E", "MCQ160F"] if c not in adults.columns]
    merged = adults if not missing_cols else adults.merge(
        mcq[["SEQN"] + missing_cols],
        on="SEQN", how="left", validate="one_to_one"
    )
    assert len(merged) == before
    mcq_adults[cycle] = merged
    mcq_summary.append({
        "cycle": cycle,
        "MCQ_already_present": not missing_cols,
        "rows_before": before,
        "rows_after": len(merged),
    })

adults_i = mcq_adults["2015-2016"]
adults_j = mcq_adults["2017-2018"]
pd.DataFrame(mcq_summary).set_index("cycle")


           MCQ_already_present  rows_before  rows_after
cycle                                                  
2015-2016                False         5992        5992
2017-2018                False         5856        5856

In [ ]:
pd.concat({
    cycle: pd.DataFrame({
        column: adults[column].value_counts(dropna=False)
        for column in ["MCQ160E", "MCQ160F"]
    }).rename_axis("response")
    for cycle, adults in [("2015-2016", adults_i), ("2017-2018", adults_j)]
}, names=["cycle"])


                    MCQ160E  MCQ160F
cycle     response                  
2015-2016 2.0          5459     5505
          NaN           273      273
          1.0           251      209
          9.0             9        5
2017-2018 2.0          5289     5286
          NaN           287      287
          1.0           270      273
          9.0            10       10

In [ ]:
mcq_age_missing = []
for cycle, adults in [("2015-2016", adults_i), ("2017-2018", adults_j)]:
    for age_group, mask in [
        ("18", adults["RIDAGEYR"].eq(18)),
        ("19", adults["RIDAGEYR"].eq(19)),
        ("20+", adults["RIDAGEYR"].ge(20)),
    ]:
        group = adults.loc[mask]
        mcq_age_missing.append({
            "cycle": cycle,
            "age_group": age_group,
            "people": len(group),
            "MCQ160E_missing": int(group["MCQ160E"].isna().sum()),
            "MCQ160F_missing": int(group["MCQ160F"].isna().sum()),
        })
pd.DataFrame(mcq_age_missing).set_index(["cycle", "age_group"])


                     people  MCQ160E_missing  MCQ160F_missing
cycle     age_group                                          
2015-2016 18            138              138              138
          19            135              135              135
          20+          5719                0                0
2017-2018 18            144              144              144
          19            143              143              143
          20+          5569                0                0

## Nối vận động

CDC PAQ_I hỏi từ 12 tuổi, PAQ_J từ 18 tuổi; chỉ dùng hoạt động giải trí, khác phần công việc và di chuyển.

Phút tương đương vừa/tuần = PAQ670 × PAD675 + 2 × PAQ655 × PAD660.

| Từng nhánh | Đóng góp phút |
|---|---|
| Trả lời Không (2) | 0, kể cả ngày/phút thiếu do skip |
| Trả lời Có (1) | Cần 1–7 ngày và 10–<1.440 phút/ngày |
| Mã khác, thiếu hoặc ngoài miền | Chưa xác định |

Gán 1 khi phần đã biết đủ ≥150 phút; gán 0 khi cả hai nhánh xác định và tổng <150; còn lại NA. Tổng phút là NA nếu một nhánh chưa rõ.

In [ ]:
paq_variables = pd.DataFrame([
    ("work_vigorous", "PAQ605", "PAQ610", "PAD615"),
    ("work_moderate", "PAQ620", "PAQ625", "PAD630"),
    ("transport_walk_cycle", "PAQ635", "PAQ640", "PAD645"),
    ("recreation_vigorous", "PAQ650", "PAQ655", "PAD660"),
    ("recreation_moderate", "PAQ665", "PAQ670", "PAD675"),
], columns=["activity", "participation", "days_per_week", "minutes_per_day"])
paq_variables


               activity participation days_per_week minutes_per_day
0         work_vigorous        PAQ605        PAQ610          PAD615
1         work_moderate        PAQ620        PAQ625          PAD630
2  transport_walk_cycle        PAQ635        PAQ640          PAD645
3   recreation_vigorous        PAQ650        PAQ655          PAD660
4   recreation_moderate        PAQ665        PAQ670          PAD675

In [ ]:
# PAQ dùng 7/9, 77/99 và 7777/9999 cho từ chối/không biết ở câu tham gia, ngày và phút.
# Thiếu câu tiếp theo có thể do skip hoặc không trả lời.
paq_cols = ["SEQN"] + paq_variables.iloc[:, 1:].to_numpy().ravel().tolist()
paq_i = load_xpt(data_dir / "2015-2016/PAQ_I.xpt", columns=paq_cols)
paq_j = load_xpt(data_dir / "2017-2018/PAQ_J.xpt", columns=paq_cols)

for paq in (paq_i, paq_j):
    assert paq["SEQN"].notna().all(), "Missing SEQN in PAQ"
    assert paq["SEQN"].is_unique, "Duplicate SEQN in PAQ"

pd.DataFrame({
    cycle: {
        "rows": len(paq),
        "SEQN_missing": int(paq["SEQN"].isna().sum()),
        "SEQN_duplicates": int(paq["SEQN"].duplicated().sum()),
    }
    for cycle, paq in [("2015-2016", paq_i), ("2017-2018", paq_j)]
})


                 2015-2016  2017-2018
rows                  9255       5856
SEQN_missing             0          0
SEQN_duplicates          0          0

In [ ]:
paq_adults = {}
paq_summary = []
for cycle, adults, paq in [
    ("2015-2016", adults_i, paq_i),
    ("2017-2018", adults_j, paq_j),
]:
    before = len(adults)
    missing_cols = [c for c in paq_cols[1:] if c not in adults.columns]
    merged = adults if not missing_cols else adults.merge(
        paq[["SEQN"] + missing_cols],
        on="SEQN", how="left", validate="one_to_one"
    )
    assert len(merged) == before
    paq_adults[cycle] = merged
    paq_summary.append({
        "cycle": cycle,
        "PAQ_already_present": not missing_cols,
        "rows_before": before,
        "rows_after": len(merged),
    })

adults_i = paq_adults["2015-2016"]
adults_j = paq_adults["2017-2018"]
pd.DataFrame(paq_summary).set_index("cycle")


           PAQ_already_present  rows_before  rows_after
cycle                                                  
2015-2016                False         5992        5992
2017-2018                False         5856        5856

In [ ]:
physical_activity_checks = []
for cycle, adults in [("2015-2016", adults_i), ("2017-2018", adults_j)]:
    before = len(adults)
    paq_before = adults[paq_cols].copy()
    activity = get_physical_activity(adults)
    adults[activity.columns] = activity
    assert len(adults) == before
    pd.testing.assert_frame_equal(adults[paq_cols], paq_before)
    physical_activity_checks.append({
        "cycle": cycle, "rows_before": before, "rows_after": len(adults)
    })
pd.DataFrame(physical_activity_checks).set_index("cycle")


           rows_before  rows_after
cycle                             
2015-2016         5992        5992
2017-2018         5856        5856

In [ ]:
pd.DataFrame({
    "2015-2016": adults_i["physical_activity"].value_counts(dropna=False),
    "2017-2018": adults_j["physical_activity"].value_counts(dropna=False),
}).reindex([1, 0, pd.NA], fill_value=0).fillna(0).astype(int)


                   2015-2016  2017-2018
physical_activity                      
1                       2113       1944
0                       3861       3903
<NA>                      18          9

In [ ]:
history_checks = []
for cycle, adults in [("2015-2016", adults_i), ("2017-2018", adults_j)]:
    before = len(adults)
    mcq_before = adults[["MCQ160E", "MCQ160F"]].copy()
    adults["heart_attack"] = get_yes_no(adults["MCQ160E"])
    adults["stroke"] = get_yes_no(adults["MCQ160F"])
    assert len(adults) == before
    pd.testing.assert_frame_equal(adults[["MCQ160E", "MCQ160F"]], mcq_before)
    assert adults.loc[
        adults["RIDAGEYR"].between(18, 19), ["heart_attack", "stroke"]
    ].isna().all().all()
    history_checks.append({
        "cycle": cycle, "rows_before": before, "rows_after": len(adults)
    })
pd.DataFrame(history_checks).set_index("cycle")


           rows_before  rows_after
cycle                             
2015-2016         5992        5992
2017-2018         5856        5856

In [ ]:
pd.concat({
    cycle: pd.DataFrame({
        column: adults[column].value_counts(dropna=False)
        for column in ["heart_attack", "stroke"]
    }).reindex([1, 0, pd.NA], fill_value=0).fillna(0).astype(int)
    for cycle, adults in [("2015-2016", adults_i), ("2017-2018", adults_j)]
}, names=["cycle", "value"])


                 heart_attack  stroke
cycle     value                      
2015-2016 1               251     209
          0              5459    5505
          <NA>            282     278
2017-2018 1               270     273
          0              5289    5286
          <NA>            297     297

## Gộp NHANES

In [ ]:
adults_i_cycle = adults_i.copy()
adults_j_cycle = adults_j.copy()
adults_i_cycle["cycle"] = "2015-2016"
adults_j_cycle["cycle"] = "2017-2018"
nhanes = pd.concat([adults_i_cycle, adults_j_cycle], ignore_index=True)

assert len(nhanes) == len(adults_i) + len(adults_j) == 11848
assert nhanes["SEQN"].notna().all()
assert nhanes["SEQN"].is_unique, "Duplicate SEQN across or within cycles"
assert set(adults_i.columns).union(adults_j.columns).issubset(nhanes.columns)

pd.DataFrame({
    "rows": nhanes.groupby("cycle").size(),
    "SEQN_duplicates": nhanes.groupby("cycle")["SEQN"].apply(
        lambda s: int(s.duplicated().sum())
    ),
})


           rows  SEQN_duplicates
cycle                           
2015-2016  5992                0
2017-2018  5856                0

In [ ]:
candidate_cols = [
    "RIDAGEYR", "RIAGENDR", "RIDRETH3", "BMXBMI",
    "smoking_status", "physical_activity", "heart_attack", "stroke",
]
X_candidate = nhanes[candidate_cols].copy()
assert X_candidate.shape == (11848, 8)

candidate_summary = pd.DataFrame({
    "variable": candidate_cols,
    "dtype": [str(X_candidate[c].dtype) for c in candidate_cols],
    "missing": [int(X_candidate[c].isna().sum()) for c in candidate_cols],
    "missing_percent": [X_candidate[c].isna().mean() * 100 for c in candidate_cols],
})
candidate_summary.round({"missing_percent": 2})


            variable    dtype  missing  missing_percent
0           RIDAGEYR  float64        0             0.00
1           RIAGENDR  float64        0             0.00
2           RIDRETH3  float64        0             0.00
3             BMXBMI  float64      752             6.35
4     smoking_status   string       11             0.09
5  physical_activity    Int64       27             0.23
6       heart_attack    Int64      579             4.89
7             stroke    Int64      575             4.85

In [ ]:
pd.Series({
    "adults_i": "glucose_usable" in adults_i.columns,
    "adults_j": "glucose_usable" in adults_j.columns,
    "nhanes": "glucose_usable" in nhanes.columns,
    **{
        f"glucose_checked[{cycle}]": "glucose_usable" in checked.columns
        for cycle, checked in glucose_checked.items()
    },
}, name="has_glucose_usable").to_frame()


                            has_glucose_usable
adults_i                                 False
adults_j                                 False
nhanes                                   False
glucose_checked[2015-2016]                True
glucose_checked[2017-2018]                True

## Ghép cờ glucose

In [ ]:
assert isinstance(glucose_checked, dict)
assert set(glucose_checked) == {"2015-2016", "2017-2018"}
glucose_flag_structure = []
for cycle, checked in glucose_checked.items():
    assert isinstance(checked, pd.DataFrame)
    assert checked["SEQN"].notna().all()
    assert checked["SEQN"].is_unique
    assert checked["glucose_usable"].dtype == pd.BooleanDtype()
    glucose_flag_structure.append({
        "cycle": cycle,
        "rows": len(checked),
        "columns": len(checked.columns),
        "glucose_usable_dtype": str(checked["glucose_usable"].dtype),
    })

glucose_flags = pd.concat([
    checked[["SEQN", "glucose_usable"]].copy()
    for checked in glucose_checked.values()
], ignore_index=True)
assert len(glucose_flags) == 11848
assert glucose_flags["SEQN"].notna().all()
assert glucose_flags["SEQN"].is_unique
pd.DataFrame(glucose_flag_structure).set_index("cycle")


           rows  columns glucose_usable_dtype
cycle                                        
2015-2016  5992       34              boolean
2017-2018  5856       34              boolean

In [ ]:
nhanes_before_glucose = nhanes.copy()
X_before_glucose = X_candidate.copy()
flag_present = "glucose_usable" in nhanes.columns
source_flag = "glucose_usable_source" if flag_present else "glucose_usable"
assert "glucose_usable_source" not in nhanes.columns
assert "_glucose_match" not in nhanes.columns
merged_glucose = nhanes.merge(
    glucose_flags.rename(columns={"glucose_usable": source_flag}),
    on="SEQN", how="left", validate="one_to_one", indicator="_glucose_match",
)
assert len(merged_glucose) == 11848
assert merged_glucose["_glucose_match"].eq("both").all(), "Unmatched SEQN"
glucose_match_counts = merged_glucose["_glucose_match"].value_counts()
if flag_present:
    pd.testing.assert_series_equal(
        merged_glucose["glucose_usable"], merged_glucose[source_flag],
        check_names=False,
    )
    merged_glucose = merged_glucose.drop(columns=source_flag)
nhanes = merged_glucose.drop(columns="_glucose_match")
assert nhanes["glucose_usable"].dtype == pd.BooleanDtype()
pd.testing.assert_frame_equal(
    nhanes[nhanes_before_glucose.columns], nhanes_before_glucose
)
pd.testing.assert_frame_equal(X_candidate, X_before_glucose)
glucose_match_counts.to_frame("people")


                people
_glucose_match        
both             11848
left_only            0
right_only           0

In [ ]:
glucose_by_cycle = pd.DataFrame({
    cycle: {
        "True": int(group["glucose_usable"].eq(True).sum()),
        "False": int(group["glucose_usable"].eq(False).sum()),
        "NA": int(group["glucose_usable"].isna().sum()),
    }
    for cycle, group in nhanes.groupby("cycle")
}).T
expected_glucose_counts = pd.DataFrame(
    [[2378, 195, 3419], [2384, 153, 3319]],
    index=["2015-2016", "2017-2018"], columns=["True", "False", "NA"],
)
pd.testing.assert_frame_equal(glucose_by_cycle, expected_glucose_counts)
glucose_by_cycle


           True  False    NA
2015-2016  2378    195  3419
2017-2018  2384    153  3319

## Tạo nhãn NHANES

Nhãn âm là giả định của dự án vì paper không mô tả đủ cách xử lý thành phần thiếu.

| Nhãn | Quy tắc |
|---|---|
| 1 | `DIQ010`, `DIQ050` hoặc `DIQ070` bằng 1; hoặc `LBXGH ≥ 6.5` (%); hoặc glucose dùng được và `LBXGLU ≥ 126` (mg/dL). Một bằng chứng dương là đủ. |
| 0 | `DIQ010` thuộc {2, 3}, có `LBXGH < 6.5` và không có bằng chứng dương hợp lệ; không bắt buộc có glucose. |
| NA | Các trường hợp còn lại. |

Borderline không tự tạo nhãn 1. `DIQ050/DIQ070` là 7/9 hoặc thiếu vẫn có thể nhận nhãn 0, nhưng thiếu do skip không phải câu trả lời Không; nhãn 0 không khẳng định chắc chắn không mắc bệnh.

In [ ]:
nhanes_before_label = nhanes.copy()
nhanes["diabetes_label"] = get_diabetes_label(nhanes)
assert nhanes["diabetes_label"].dtype == pd.Int64Dtype()
assert len(nhanes) == len(nhanes_before_label) == 11848
pd.testing.assert_frame_equal(
    nhanes[nhanes_before_label.columns], nhanes_before_label
)
pd.DataFrame({
    "rows_before": nhanes_before_label.groupby("cycle").size(),
    "rows_after": nhanes.groupby("cycle").size(),
})


           rows_before  rows_after
cycle                             
2015-2016         5992        5992
2017-2018         5856        5856

In [ ]:
diabetes_label_counts = pd.DataFrame({
    cycle: group["diabetes_label"].value_counts(dropna=False)
    for cycle, group in nhanes.groupby("cycle")
}).reindex([1, 0, pd.NA], fill_value=0).fillna(0).astype(int)
assert diabetes_label_counts.loc[1].tolist() == [1105, 1169]
assert diabetes_label_counts.sum().tolist() == [5992, 5856]
diabetes_label_counts


                2015-2016  2017-2018
diabetes_label                      
1                    1105       1169
0                    4397       4161
<NA>                  490        526

In [ ]:
analysis_inputs = [
    "RIDAGEYR", "RIAGENDR", "RIDRETH3", "BMXBMI",
    "smoking_status", "physical_activity", "heart_attack", "stroke",
]
survey_cols = ["WTINT2YR", "WTMEC2YR", "WTSAF2YR", "SDMVPSU", "SDMVSTRA"]
analysis_cols = ["SEQN", "cycle"] + analysis_inputs + ["diabetes_label"] + survey_cols
analysis_data = nhanes.loc[nhanes["diabetes_label"].notna(), analysis_cols].copy()

assert len(analysis_data) == 10832
assert analysis_data["SEQN"].notna().all()
assert analysis_data["SEQN"].is_unique
assert analysis_data["diabetes_label"].isin([0, 1]).all()
pd.Series({
    "people": len(analysis_data),
    "columns": len(analysis_data.columns),
    "SEQN_duplicates": int(analysis_data["SEQN"].duplicated().sum()),
}, name="analysis_data").to_frame()


                 analysis_data
people                   10832
columns                     16
SEQN_duplicates              0

In [ ]:
analysis_label_counts = pd.crosstab(
    analysis_data["cycle"], analysis_data["diabetes_label"]
).reindex(columns=[0, 1], fill_value=0)
analysis_label_counts.assign(total=analysis_label_counts.sum(axis=1))


diabetes_label     0     1  total
cycle                            
2015-2016       4397  1105   5502
2017-2018       4161  1169   5330

In [ ]:
analysis_input_missing = pd.DataFrame({
    "variable": analysis_inputs,
    "missing": [int(analysis_data[c].isna().sum()) for c in analysis_inputs],
    "missing_percent": [analysis_data[c].isna().mean() * 100 for c in analysis_inputs],
})
analysis_input_missing.round({"missing_percent": 2})


            variable  missing  missing_percent
0           RIDAGEYR        0             0.00
1           RIAGENDR        0             0.00
2           RIDRETH3        0             0.00
3             BMXBMI      228             2.10
4     smoking_status        9             0.08
5  physical_activity       25             0.23
6       heart_attack      499             4.61
7             stroke      495             4.57

## Đầu vào chung

Nguồn mã: CDC DEMO_I/J, SMQ_I/J, MCQ_I/J, PAQ_I/J và codebook BRFSS 2015/2017.

| Biến | NHANES | BRFSS |
|---|---|---|
| age | `RIDAGEYR` | `_AGE80`, gồm tuổi CDC đã điền thiếu |
| sex | `RIAGENDR`: 1 Nam, 2 Nữ | `SEX`: 1 Nam, 2 Nữ; 9 → NA |
| race_ethnicity | `RIDRETH3`: 3 White NH, 4 Black NH, 1/2 Hispanic, 6/7 Other; mã 5 không dùng | `_RACEGR3`: 1 White NH, 2 Black NH, 5 Hispanic, 3/4 Other; 9 → NA |
| bmi | `BMXBMI`, đo trực tiếp | `_BMI5 / 100`, từ chiều cao/cân nặng tự khai |
| smoking_status | `SMQ020=2` → never; `SMQ020=1` và `SMQ040=3` → former, `SMQ040∈{1,2}` → current | `_SMOKER3`: 1/2 current, 3 former, 4 never; 9 → NA |
| physical_activity | Vận động giải trí, quy tắc ở trên | `_PA150R2`: 1 → 1, 2/3 → 0, 9 → NA; hoạt động ngoài công việc trong tháng qua |
| heart_attack / stroke | `MCQ160E/F`: 1 → 1, 2 → 0; 7/9 → NA | `CVDINFR4/CVDSTRK3`: cùng quy tắc |

Tuổi hai nguồn chặn trên ở 80 (nhóm ≥80); BMI có đơn vị kg/m². Giá trị thiếu và mã không xác định vẫn là NA; `_PSU` là đơn vị khảo sát, không phải mã người.

In [ ]:
brfss_dir = Path("../data/raw/brfss")
brfss_input_cols = [
    "_AGE80", "SEX", "_RACEGR3", "_BMI5", "_SMOKER3", "_PA150R2",
    "CVDINFR4", "CVDSTRK3",
]
brfss_support_cols = ["_STATE", "_PSU", "_STSTR", "_LLCPWT", "DIABETE3"]
brfss_cols = brfss_input_cols + brfss_support_cols
brfss_raw = {}
for year in [2015, 2017]:
    with pd.read_sas(
        brfss_dir / str(year) / f"LLCP{year}.XPT", format="xport", chunksize=20000
    ) as reader:
        brfss_raw[year] = pd.concat(
            [chunk[brfss_cols].copy() for chunk in reader], ignore_index=True
        )
pd.DataFrame({year: {"rows": len(raw), "columns": len(raw.columns)}
              for year, raw in brfss_raw.items()}).T


        rows  columns
2015  441456       13
2017  450016       13

In [ ]:
nhanes_inputs = map_nhanes_inputs(analysis_data)
brfss_inputs = {year: map_brfss_inputs(raw) for year, raw in brfss_raw.items()}
brfss_support = {
    year: raw[brfss_support_cols].copy().assign(year=year)
    for year, raw in brfss_raw.items()
}
common_input_tables = {
    **{f"NHANES {cycle}": nhanes_inputs.loc[group.index].copy()
       for cycle, group in analysis_data.groupby("cycle")},
    **{f"BRFSS {year}": inputs for year, inputs in brfss_inputs.items()},
}
common_cols = [
    "age", "sex", "race_ethnicity", "bmi", "smoking_status",
    "physical_activity", "heart_attack", "stroke",
]
assert nhanes_inputs.columns.tolist() == common_cols
assert len(nhanes_inputs) == len(analysis_data)
assert analysis_data["RIDAGEYR"].max() == nhanes_inputs["age"].max() == 80
for year, inputs in brfss_inputs.items():
    assert inputs.columns.tolist() == common_cols
    assert len(inputs) == len(brfss_raw[year]) == len(brfss_support[year])
    assert inputs.index.equals(brfss_support[year].index)
    pd.testing.assert_frame_equal(
        brfss_support[year][brfss_support_cols], brfss_raw[year][brfss_support_cols]
    )
pd.DataFrame({name: {"rows": len(inputs), "columns": len(inputs.columns)}
              for name, inputs in common_input_tables.items()}).T


                    rows  columns
NHANES 2015-2016    5502        8
NHANES 2017-2018    5330        8
BRFSS 2015        441456        8
BRFSS 2017        450016        8

In [ ]:
common_domains = {
    "sex": [1, 2],
    "race_ethnicity": ["White NH", "Black NH", "Hispanic", "Other"],
    "smoking_status": ["current", "former", "never"],
    "physical_activity": [0, 1], "heart_attack": [0, 1], "stroke": [0, 1],
}
common_category_counts = {}
for name, inputs in common_input_tables.items():
    rows = []
    for column, values in common_domains.items():
        assert inputs[column].dropna().isin(values).all(), column
        counts = inputs[column].value_counts(dropna=False)
        for value in values:
            rows.append((column, str(value), int(counts.get(value, 0))))
        rows.append((column, "NA", int(inputs[column].isna().sum())))
    common_category_counts[name] = pd.DataFrame(
        rows, columns=["variable", "value", "people"]
    ).set_index(["variable", "value"])["people"]
pd.DataFrame(common_category_counts)


                            NHANES 2015-2016  ...  BRFSS 2017
variable          value                       ...            
sex               1                     2659  ...      198725
                  2                     2843  ...      251007
                  NA                       0  ...         284
race_ethnicity    White NH              1779  ...      337166
                  Black NH              1150  ...       35765
                  Hispanic              1719  ...       37078
                  Other                  854  ...       31252
                  NA                       0  ...        8755
smoking_status    current               1006  ...       63466
                  former                1235  ...      121722
                  never                 3252  ...      246104
                  NA                       9  ...       18724
physical_activity 0                     3558  ...      193844
                  1                     1927  ...      206015
        

In [ ]:
common_missing = pd.DataFrame({
    name: inputs.isna().sum() for name, inputs in common_input_tables.items()
})
common_missing


                   NHANES 2015-2016  NHANES 2017-2018  BRFSS 2015  BRFSS 2017
age                               0                 0           0           0
sex                               0                 0           0         284
race_ethnicity                    0                 0        7437        8755
bmi                              99               129       36398       36446
smoking_status                    9                 0       17990       18724
physical_activity                17                 8       53260       50157
heart_attack                    246               253        2229        2366
stroke                          242               253        1290        1350

In [ ]:
common_ranges = pd.DataFrame({
    name: {
        "age_min": inputs["age"].min(), "age_max": inputs["age"].max(),
        "bmi_min": inputs["bmi"].min(), "bmi_max": inputs["bmi"].max(),
    }
    for name, inputs in common_input_tables.items()
}).T
assert common_ranges["age_min"].ge(18).all()
assert common_ranges["age_max"].eq(80).all()
common_ranges


                  age_min  age_max  bmi_min  bmi_max
NHANES 2015-2016     18.0     80.0    14.50    67.30
NHANES 2017-2018     18.0     80.0    14.20    86.20
BRFSS 2015           18.0     80.0    12.02    99.95
BRFSS 2017           18.0     80.0    12.01    99.31

## Tạo nhãn BRFSS

| DIABETE3 | Nhãn |
|---|---|
| 1: Có | 1 |
| 2: Chỉ trong thai kỳ; 3: Không; 4: Tiền đái tháo đường/borderline | 0 |
| 7/9 hoặc thiếu | NA |

Nhãn 0 không loại trừ bệnh chưa được chẩn đoán. Bảng phân tích chỉ bỏ nhãn NA, vẫn gồm người thiếu đầu vào.

In [ ]:
brfss_labels = {}
brfss_analysis = {}
brfss_label_summary = []
for year in [2015, 2017]:
    raw = brfss_raw[year]
    inputs = brfss_inputs[year]
    support = brfss_support[year]
    assert raw.index.is_unique and inputs.index.is_unique and support.index.is_unique
    assert inputs.index.equals(support.index) and inputs.index.equals(raw.index)
    pd.testing.assert_frame_equal(inputs, map_brfss_inputs(raw))
    pd.testing.assert_frame_equal(support, raw[brfss_support_cols].assign(year=year))
    label = get_brfss_diabetes_label(support["DIABETE3"])
    assert label.index.equals(inputs.index) and label.dtype == pd.Int64Dtype()
    combined = inputs.join(support, how="left", validate="one_to_one")
    combined["diabetes_label"] = label
    known_label = label.notna()
    brfss_labels[year] = label
    brfss_analysis[year] = combined.loc[known_label].copy()
    analysis = brfss_analysis[year]
    pd.testing.assert_frame_equal(analysis[common_cols], inputs.loc[known_label])
    pd.testing.assert_series_equal(analysis["DIABETE3"], support.loc[known_label, "DIABETE3"])
    assert analysis["diabetes_label"].notna().all()
    brfss_label_summary.append({
        "year": year,
        "before": len(inputs),
        "label_1": int(label.eq(1).sum()),
        "label_0": int(label.eq(0).sum()),
        "label_NA": int(label.isna().sum()),
        "after": len(analysis),
        "positive_percent": analysis["diabetes_label"].mean() * 100,
        "retained_missing_inputs": int(analysis[common_cols].isna().any(axis=1).sum()),
    })
pd.DataFrame(brfss_label_summary).set_index("year").round({"positive_percent": 4})


      before  label_1  ...  positive_percent  retained_missing_inputs
year                   ...                                           
2015  441456    57256  ...           12.9933                    83880
2017  450016    60440  ...           13.4545                    82115

[2 rows x 7 columns]

In [ ]:
pd.DataFrame({
    year: support["DIABETE3"].value_counts(dropna=False)
    for year, support in brfss_support.items()
}).rename_axis("DIABETE3").sort_index()


            2015    2017
DIABETE3                
1.0        57256   60440
2.0         3608    3283
3.0       372104  377264
4.0         7690    8232
7.0          598     594
9.0          193     199
NaN            7       4

In [ ]:
pd.DataFrame({
    year: label.value_counts(dropna=False)
    for year, label in brfss_labels.items()
}).reindex([1, 0, pd.NA], fill_value=0).fillna(0).astype(int).rename_axis("diabetes_label")


                  2015    2017
diabetes_label                
1                57256   60440
0               383402  388779
<NA>               798     797

## Xuất dữ liệu

In [ ]:
assert nhanes_inputs.index.is_unique and analysis_data.index.is_unique
assert nhanes_inputs.index.equals(analysis_data.index)
pd.testing.assert_frame_equal(nhanes_inputs, map_nhanes_inputs(analysis_data))
nhanes_export = nhanes_inputs.join(
    analysis_data[["diabetes_label", "SEQN", "cycle"] + survey_cols],
    how="left", validate="one_to_one",
)
assert len(nhanes_export) == len(analysis_data) == 10832 
pd.testing.assert_frame_equal(nhanes_export[common_cols], nhanes_inputs)
pd.testing.assert_frame_equal(
    nhanes_export[["diabetes_label", "SEQN", "cycle"] + survey_cols],
    analysis_data[["diabetes_label", "SEQN", "cycle"] + survey_cols],
) 
export_tables = {
    "nhanes_2015_2018.parquet": nhanes_export,
    "brfss_2015.parquet": brfss_analysis[2015],
    "brfss_2017.parquet": brfss_analysis[2017],
}
pd.DataFrame({name: {"rows": len(frame), "columns": len(frame.columns)}
              for name, frame in export_tables.items()}).T


                            rows  columns
nhanes_2015_2018.parquet   10832       16
brfss_2015.parquet        440658       15
brfss_2017.parquet        449219       15

In [ ]:
import importlib.util
if importlib.util.find_spec("pyarrow") is None:
    raise ImportError("Parquet export requires pyarrow; no files have been written.")

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)
parquet_checks = []
for name, frame in export_tables.items():
    path = processed_dir / name
    if not path.exists():
        frame.to_parquet(path, engine="pyarrow", index=True)
    restored = pd.read_parquet(path, engine="pyarrow")
    pd.testing.assert_frame_equal(restored, frame, check_dtype=True, check_exact=True)
    assert restored.shape == frame.shape
    parquet_checks.append({
        "path": str(path), "rows": len(restored), "columns": len(restored.columns),
        "roundtrip_exact": True,
    })
pd.DataFrame(parquet_checks)


                                         path    rows  columns  roundtrip_exact
0  ..\data\processed\nhanes_2015_2018.parquet   10832       16             True
1        ..\data\processed\brfss_2015.parquet  440658       15             True
2        ..\data\processed\brfss_2017.parquet  449219       15             True